## 08_lenormand_graph_export
  Exports the observed and the mean simulated networks of 07/07b as weighted directed graphml.
  No model is rerun. Environment: nuts3_tdlm.

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
COL_ID, COL_POP = "nuts3_id", "population"
GISCO_GEOJSON = config.RAW_DIR / "nuts3" / "NUTS_RG_03M_2024_4326.geojson"
GISCO_COL_ID, GISCO_COL_NAME = "NUTS_ID", "NUTS_NAME"
LAWS = ["NGravExp", "NGravPow", "Rad"]
CONSTRAINT_LEVELS = ["UM", "PCM", "ACM", "DCM"]
OUT_DIR = config.PROCESSED_DIR / "08_lenormand_graph_export"

# Per country: shared inputs, the folder of the simulated matrices, and the variants
# (variant name -> observed OD file, simulated matrix file name pattern)
SOURCES = {
    "CZ": {
        "shared_dir": config.INTERIM_DIR / "lenormand_comparison" / "cz",
        "runs_dir": config.PROCESSED_DIR / "07_lenormand_pytdlm_cz" / "runs",
        "variants": {"observed_only": ("od_observed.csv", "sim_{law}_{level}.npy")},
    },
    "PL": {
        "shared_dir": config.INTERIM_DIR / "lenormand_comparison" / "pl",
        "runs_dir": config.PROCESSED_DIR / "07b_lenormand_pytdlm_pl" / "runs",
        "variants": {
            "missing0": ("od_observed_missing0.csv", "sim_missing0_{law}_{level}.npy"),
            "missing2": ("od_observed_missing2.csv", "sim_missing2_{law}_{level}.npy"),
        },
    },
}
# ------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Region names from the GISCO file (properties only)
with open(GISCO_GEOJSON, encoding="utf-8") as f:
    props = pd.DataFrame([ft["properties"] for ft in json.load(f)["features"]])
code_to_name = dict(zip(props[GISCO_COL_ID], props[GISCO_COL_NAME]))

nodes = {}
for country, src in SOURCES.items():
    pop = pd.read_csv(src["shared_dir"] / "population.csv")
    pop["name"] = pop[COL_ID].map(code_to_name)
    nodes[country] = pop
    n_files_expected = len(LAWS) * len(CONSTRAINT_LEVELS)
    print(f"{country}: {len(pop)} regions | without name: {int(pop['name'].isna().sum())}")
    for variant, (od_file, sim_pattern) in src["variants"].items():
        od_ok = (src["shared_dir"] / od_file).exists()
        n_sim = sum((src["runs_dir"] / sim_pattern.format(law=law, level=level)).exists()
                    for law in LAWS for level in CONSTRAINT_LEVELS)
        print(f"  {variant}: observed OD file found: {od_ok} | "
              f"simulated files found: {n_sim} of {n_files_expected}")

In [ ]:
# ----------------------------- CONFIG -----------------------------
CHECK_LAW, CHECK_LEVEL = "NGravExp", "PCM"
# ------------------------------------------------------------------

def load_observed(shared_dir, od_file, ids):
    """Read an observed OD matrix and check that its order matches the region ids."""
    od = pd.read_csv(shared_dir / od_file, index_col=0)
    od.index = od.index.astype(str)
    od.columns = od.columns.astype(str)
    assert list(od.index) == list(ids), "row order differs from population.csv"
    assert list(od.columns) == list(ids), "column order differs from population.csv"
    return od.to_numpy(dtype=float)

def load_simulated_mean(path):
    """Load a simulated matrix file; average over replications if it is 3D."""
    arr = np.load(path)
    n_rep = arr.shape[0] if arr.ndim == 3 else 1
    return (arr.mean(axis=0) if arr.ndim == 3 else arr), arr.shape, n_rep

for country, src in SOURCES.items():
    ids = nodes[country][COL_ID].astype(str).tolist()
    for variant, (od_file, sim_pattern) in src["variants"].items():
        obs = load_observed(src["shared_dir"], od_file, ids)
        sim_path = src["runs_dir"] / sim_pattern.format(law=CHECK_LAW, level=CHECK_LEVEL)
        sim_mean, raw_shape, n_rep = load_simulated_mean(sim_path)
        print(f"{country}/{variant}")
        print(f"  observed: shape {obs.shape} | diagonal sum {np.trace(obs):.0f} | "
              f"total {obs.sum():.0f} | non-zero cells {int((obs > 0).sum())}")
        print(f"  simulated {CHECK_LAW}/{CHECK_LEVEL}: raw shape {raw_shape} | "
              f"replications {n_rep} | mean-matrix total {sim_mean.sum():.0f} | "
              f"diagonal sum {np.trace(sim_mean):.0f} | non-zero cells {int((sim_mean > 0).sum())}")

In [ ]:
import networkx as nx

# ----------------------------- CONFIG -----------------------------
SUMMARY_FILE = OUT_DIR / "graph_summary.csv"
# ------------------------------------------------------------------

def build_graph(matrix, nodes_df, attrs):
    """Directed weighted graph from a flow matrix; diagonal removed, zero cells skipped."""
    m = np.array(matrix, dtype=float)
    np.fill_diagonal(m, 0.0)  # no self-loops
    ids = nodes_df[COL_ID].astype(str).tolist()
    g = nx.from_numpy_array(m, create_using=nx.DiGraph)
    g = nx.relabel_nodes(g, dict(enumerate(ids)))
    for node_id, name, pop_value in zip(ids, nodes_df["name"], nodes_df[COL_POP]):
        g.nodes[node_id]["name"] = str(name)
        g.nodes[node_id]["population"] = float(pop_value)
    g.graph.update(attrs)
    return g

rows = []
for country, src in SOURCES.items():
    nodes_df = nodes[country]
    ids = nodes_df[COL_ID].astype(str).tolist()
    for variant, (od_file, sim_pattern) in src["variants"].items():
        # Observed graph plus the 12 mean simulated graphs of this variant
        jobs = [("observed", "observed", load_observed(src["shared_dir"], od_file, ids), 0)]
        for law in LAWS:
            for level in CONSTRAINT_LEVELS:
                mean_matrix, _, n_rep = load_simulated_mean(
                    src["runs_dir"] / sim_pattern.format(law=law, level=level))
                jobs.append((law, level, mean_matrix, n_rep))
        for law, level, matrix, n_rep in jobs:
            attrs = {"country": country, "variant": variant, "law": law,
                     "constraint_level": level, "n_replications": int(n_rep)}
            g = build_graph(matrix, nodes_df, attrs)
            fname = f"{country}_{variant}_{law}_{level}.graphml"
            if not (OUT_DIR / fname).exists():
                nx.write_graphml(g, OUT_DIR / fname)
            rows.append({"file": fname, "country": country, "variant": variant,
                         "law": law, "constraint_level": level, "n_replications": n_rep,
                         "n_nodes": g.number_of_nodes(), "n_edges": g.number_of_edges(),
                         "total_weight": g.size(weight="weight")})

graph_summary = pd.DataFrame(rows)
graph_summary.to_csv(SUMMARY_FILE, index=False)
print(f"graphs: {len(graph_summary)} | folder: {OUT_DIR.parent.name}/{OUT_DIR.name}")
print(graph_summary.drop(columns=["file"]).round(0).to_string(index=False))

In [ ]:
# ----------------------------- CONFIG -----------------------------
REP_INDEX = 0  # which replication to export (0..n_replications-1)
SINGLE_DIR = OUT_DIR / "single_replication"
SINGLE_SUMMARY_FILE = OUT_DIR / "graph_summary_single_replication.csv"
# ------------------------------------------------------------------

SINGLE_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for country, src in SOURCES.items():
    nodes_df = nodes[country]
    for variant, (od_file, sim_pattern) in src["variants"].items():
        for law in LAWS:
            for level in CONSTRAINT_LEVELS:
                arr = np.load(src["runs_dir"] / sim_pattern.format(law=law, level=level))
                assert arr.ndim == 3 and REP_INDEX < arr.shape[0], "REP_INDEX out of range"
                attrs = {"country": country, "variant": variant, "law": law,
                         "constraint_level": level, "n_replications": 1,
                         "replication_index": int(REP_INDEX)}
                g = build_graph(arr[REP_INDEX], nodes_df, attrs)
                fname = f"{country}_{variant}_{law}_{level}_rep{REP_INDEX}.graphml"
                if not (SINGLE_DIR / fname).exists():
                    nx.write_graphml(g, SINGLE_DIR / fname)
                rows.append({"file": fname, "country": country, "variant": variant,
                             "law": law, "constraint_level": level,
                             "replication_index": REP_INDEX,
                             "n_nodes": g.number_of_nodes(), "n_edges": g.number_of_edges(),
                             "total_weight": g.size(weight="weight")})

single_summary = pd.DataFrame(rows)
single_summary.to_csv(SINGLE_SUMMARY_FILE, index=False)
print(f"graphs: {len(single_summary)} | folder: {SINGLE_DIR.parent.name}/{SINGLE_DIR.name}")
print(single_summary.drop(columns=["file"]).round(0).to_string(index=False))